In [1]:
import pandas as pd  
import numpy as np
import re
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from sklearn.model_selection import train_test_split  
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from math import sqrt
from sklearn.covariance import EllipticEnvelope  
from sklearn.ensemble import RandomForestRegressor  
from sklearn.metrics import mean_absolute_error, mean_squared_error  

In [2]:

def get_ratio(text):
    text = str(text)  
    
    
    parts = re.findall(r'(\d+|[一二三四五六七八九十百]+)', text)
    
    if len(parts) == 2: 
        try:
           
            num_elevators = chinese_to_arabic(parts[0])  
            num_households = chinese_to_arabic(parts[1]) 
            
            if num_elevators > 0: 
                return num_households / num_elevators  
        except: 
            return np.nan  
    
    return np.nan  

def classify_common_floor(row):
   
    if row['楼层类型'] != '普通楼层':
        return row['楼层类型']
    
   
    try:
        current_floor = int(row['初始楼层类型'])
        total_floors = row['总楼层']
        
        if pd.notna(total_floors) and total_floors > 0:
            ratio = current_floor / total_floors
            if ratio < 0.33:
                return '低楼层'
            elif ratio < 0.66:
                return '中楼层'
            else:
                return '高楼层'
    except (ValueError, TypeError):
        pass 
    return '普通楼层'


In [3]:
# 1. 数据处理
data_rent =r"D:\人工智能\Python exam\ruc_Class25Q2_train_rent.csv"
df1 = pd.read_csv(data_rent, dtype=str)
#print("===== df1 原始数据基本信息 =====")
#print(df1.info()) 
non_null_counts_df1 = df1.notnull().sum()
keep_cols_df1 = non_null_counts_df1[non_null_counts_df1 > 70000].index.tolist()
df1 = df1[keep_cols_df1]
#print("\n===== df1 筛选后数据基本信息 =====")
#print(df1.info())
#df1 = df1.dropna()  
#df1 = df1.drop_duplicates()  
#print(df1.info())
# 创建核心特征 '房龄' 
df1['交易时间'] = pd.to_datetime(df1['交易时间'], errors='coerce')
df1['交易年份'] = df1['交易时间'].dt.year

def parse_build_year(year_str):
   
    year_str = str(year_str)
   
    years = re.findall(r'\d{4}', year_str)
    
    if years:
        
        return np.mean([int(y) for y in years])
    
    return np.nan 


df1['建筑年份'] = df1['建筑年代'].apply(parse_build_year)

df1['房龄'] = df1['交易年份'] - df1['建筑年份']

df1['卧室数'] = df1['户型'].astype(str).str.extract(r'(\d+)[室房]').fillna(0).astype(int)
df1['客厅数'] = df1['户型'].astype(str).str.extract(r'(\d+)厅').fillna(0).astype(int)
df1['卫生间数'] = df1['户型'].astype(str).str.extract(r'(\d+)卫').fillna(0).astype(int)


s = df1['楼层'].astype(str)
df1['总楼层'] = df1['楼层'].astype(str).str.extract(r'(\d+)层').squeeze()

df1['总楼层'] = pd.to_numeric(df1['总楼层'], errors='coerce')
df1['初始楼层类型'] = s.str.extract(r'^(.*?)/').squeeze()


df1['楼层类型'] = s.str.extract(r'^(低楼层|中楼层|高楼层)').squeeze()
df1['楼层类型'] = df1['楼层类型'].fillna('普通楼层')

df1['楼层类型'] = df1.apply(classify_common_floor, axis=1)
numeric_cols = ['Price', '面积', 'lon', 'lat', 'coord_x', 'coord_y','城市','房屋总数','楼栋总数','绿 化 率','容 积 率','物 业 费','燃气费','停车位','停车费用']

def process_range_text(text):
    
    text_str = str(text)
    cleaned_text = re.sub(r"[^\d.-]", "", text_str)

    if "-" in cleaned_text:
        
        num1, num2 = cleaned_text.split("-")
        return (float(num1) + float(num2)) / 2
    
    else:
        
        return float(cleaned_text) if cleaned_text.strip() else np.nan

for col in numeric_cols:
    if col == 'Price':
        
        df1[col] = df1[col].str.replace('¥', '').str.replace(',', '')
        df1[col] = df1[col].apply(process_range_text)
        
        #df1[col] = df1[col] / 1000000
    
    elif col == '面积':
        
        df1[col] = df1[col].str.replace('㎡', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '房屋总数':
        
        df1[col] = df1[col].str.replace('户', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '楼栋总数':
        
        df1[col] = df1[col].str.replace('栋', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '绿化率':
       
        df1[col] = df1[col].str.replace('%', '')
        df1[col] = df1[col].apply(process_range_text)
        df1[col] = df1[col] / 100 
    
    else:
        
        df1[col] = df1[col].apply(process_range_text)
df1['户均楼栋房屋数'] = df1['房屋总数'] / df1['楼栋总数']
df1['每户停车位'] = df1['停车位'] / df1['房屋总数']

df1['朝向'] = df1['朝向'].astype(str).str.replace(' ', '')  


base_directions = ['东', '南', '西', '北']
for direction in base_directions:

    df1[f'朝向_{direction}'] = df1['朝向'].str.contains(direction, na=False).astype(int)


df1['朝向'] = (df1['朝向'] == '未知').astype(int)


if all(col in df1.columns for col in ['核心卖点', '户型介绍', '周边配套']):
        df1['description_combined'] = (
            df1['核心卖点'].fillna('') + 
            df1['户型介绍'].fillna('') + 
            df1['周边配套'].fillna('')
        )
        

        objective_keywords = ['户型方正', '人车分流', '学区', '地铁', '医院', '商场', '超市', '公园', '菜市场']
        for keyword in objective_keywords:
            df1[f'Desc_{keyword}'] = df1['description_combined'].str.contains(keyword, na=False).astype(int)

if '客户反馈' in df1.columns:
        positive_keywords = ['体验佳', '干净', '安静', '方便', '采光好', '物业好', '安全','好', '安全', '阳光充足', '整洁']
        negative_keywords = ['老旧', '费高', '噪音', '通风差', '潮湿', '漏水', '老化', '乱', '卫生差','一般']
        
        df1['积极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in positive_keywords if word in x)
        )
        df1['消极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in negative_keywords if word in x)
        )
        df1['综合反馈'] = df1['积极反馈'] - df1['消极反馈']

numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()
df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())
print("===== 数值列转换后信息 =====")
print(df1.info())
#print(df1.head(15)) 

#print("\n数据统计描述：")
#print(df1.describe())  
#print("\n前5行数据：")
#print(df1.head()) 

===== 数值列转换后信息 =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 55 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   城市       98899 non-null  float64       
 1   户型       98898 non-null  object        
 2   Price    98899 non-null  float64       
 3   楼层       98894 non-null  object        
 4   面积       98899 non-null  float64       
 5   朝向       98899 non-null  int64         
 6   交易时间     98899 non-null  datetime64[ns]
 7   付款方式     80476 non-null  object        
 8   租赁方式     98899 non-null  object        
 9   电梯       98895 non-null  object        
 10  用水       81159 non-null  object        
 11  用电       81575 non-null  object        
 12  燃气       94317 non-null  object        
 13  lon      98899 non-null  float64       
 14  lat      98899 non-null  float64       
 15  年份       98899 non-null  object        
 16  区县       94222 non-null  object        
 17  板块       9

In [4]:
# 确定特征（X）和目标变量（y）
X_rent = df1.drop(columns=['Price']) 
y_rent = df1['Price'] 
X_train_rent, X_test_rent, y_train_rent, y_test_rent = train_test_split(
    X_rent, y_rent,
    test_size=0.3,  
    random_state=111  
)
df1 = df1.loc[:, ~df1.columns.duplicated()]  

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

df1['Price'] = np.log(df1['Price'])  
df1['面积'] = np.log(df1['面积'])   


numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()

X = df1.drop('Price', axis=1)  
y = df1['Price']               
numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

one_hot_cols = [col for col in numeric_cols if X[col].nunique() <= 2]  
continuous_cols = [col for col in numeric_cols if col not in one_hot_cols]  

preprocessor = ColumnTransformer(
    transformers=[
        
        ('num', StandardScaler(), continuous_cols),
       
        ('cat', 'passthrough', one_hot_cols)
    ],
    remainder='drop'  
)

X_processed = preprocessor.fit_transform(X)  

processed_columns = continuous_cols + one_hot_cols  
X_processed_df = pd.DataFrame(X_processed, columns=processed_columns, index=X.index)


print(X_processed_df.head())

        城市        面积       lon       lat      房屋总数      楼栋总数     绿 化 率  \
0 -1.30137 -1.073219  0.284010  1.490919 -0.155475 -0.153682 -0.041799   
1 -1.30137 -0.875238  0.304326  1.482347 -0.049104 -0.329328 -0.041799   
2 -1.30137 -1.030628  0.316216  1.486631 -0.070378 -0.276634 -0.041799   
3 -1.30137 -0.371538  0.344314  1.521390  0.533281 -0.311763 -0.026219   
4 -1.30137 -0.567117  0.289052  1.500328  0.535940 -0.013166 -0.057379   

      容 积 率     物 业 费       燃气费  ...   户均楼栋房屋数     每户停车位      积极反馈      消极反馈  \
0 -0.303889 -0.355115 -0.570920  ... -0.466143 -0.251149  1.075698 -0.528577   
1 -1.152185 -0.529788 -0.570920  ...  0.288054 -0.349368 -0.611835  1.563121   
2 -0.173382 -0.106206 -0.551268  ... -0.060007 -0.116257 -0.611835 -0.528577   
3 -0.108129  0.147069 -0.570920  ...  0.825946 -0.302192 -0.611835 -0.528577   
4 -0.825917 -0.493397 -0.570920  ... -0.336907 -0.320116 -0.611835 -0.528577   

       综合反馈   朝向  朝向_东  朝向_南  朝向_西  朝向_北  
0  1.111903  0.0   0.0   0.0   

In [6]:

if 'X_processed_df' in locals():  
    print("\n==== 预处理后特征矩阵各列缺失值统计 ====")
    processed_missing = X_processed_df.isnull().sum().reset_index()
    processed_missing.columns = ["列名", "缺失值数量"]
    processed_missing["是否存在缺失"] = processed_missing["缺失值数量"] > 0
    print(processed_missing)


==== 预处理后特征矩阵各列缺失值统计 ====
         列名  缺失值数量  是否存在缺失
0        城市      0   False
1        面积      0   False
2       lon      0   False
3       lat      0   False
4      房屋总数      0   False
5      楼栋总数      0   False
6     绿 化 率      0   False
7     容 积 率      0   False
8     物 业 费      0   False
9       燃气费      0   False
10      停车位      0   False
11     停车费用      0   False
12  coord_x      0   False
13  coord_y      0   False
14     建筑年份      0   False
15       房龄      0   False
16      卧室数      0   False
17      客厅数      0   False
18     卫生间数      0   False
19      总楼层      0   False
20  户均楼栋房屋数      0   False
21    每户停车位      0   False
22     积极反馈      0   False
23     消极反馈      0   False
24     综合反馈      0   False
25       朝向      0   False
26     朝向_东      0   False
27     朝向_南      0   False
28     朝向_西      0   False
29     朝向_北      0   False


In [7]:

X = X_processed  
y = df1['Price'].values  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=111  
)
print(f"训练集样本数: {X_train.shape[0]}, 测试集样本数: {X_test.shape[0]}")



# 计算训练集目标变量的IQR
Q1 = np.percentile(y_train, 25)
Q3 = np.percentile(y_train, 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 筛选训练集非异常值样本
train_mask = (y_train >= lower_bound) & (y_train <= upper_bound)
X_train_clean = X_train[train_mask]
y_train_clean = y_train[train_mask]
print(f"异常值处理前训练集样本数: {X_train.shape[0]} → 处理后: {X_train_clean.shape[0]}")



# 3.1 生成二次多项式特征（含交互项）
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_clean)  
X_test_poly = poly.transform(X_test)  


训练集样本数: 79119, 测试集样本数: 19780
异常值处理前训练集样本数: 79119 → 处理后: 78552


In [8]:

from sklearn.feature_selection import SelectFromModel
# 3.2 用Lasso做特征选择
lasso_selector = Lasso(alpha=0.01, random_state=111)
lasso_selector.fit(X_train_poly, y_train_clean)
selector = SelectFromModel(lasso_selector, prefit=True)

# 筛选后的特征
X_train_selected = selector.transform(X_train_poly)
X_test_selected = selector.transform(X_test_poly)
print(f"特征选择前维度: {X_train_poly.shape[1]} → 选择后: {X_train_selected.shape[1]}")


特征选择前维度: 495 → 选择后: 81


In [9]:

original_feature_names = df1.drop('Price', axis=1).columns.tolist()
# ---------- 查看Lasso选择后的具体特征变量 ----------
# 1. 获取多项式特征的完整列名
poly_feature_names = poly.get_feature_names_out(input_features=processed_columns)

# 2. 获取特征选择的掩码
selected_mask = selector.get_support()

# 3. 过滤得到被选中的特征名
selected_features = poly_feature_names[selected_mask]

print("\n==== Lasso选择后的特征变量 ====")
for idx, feat in enumerate(selected_features, 1):
    print(f"{idx}. {feat}")
    


==== Lasso选择后的特征变量 ====
1. 城市
2. 面积
3. lon
4. lat
5. 房屋总数
6. 物 业 费
7. 燃气费
8. 停车位
9. 房龄
10. 客厅数
11. 总楼层
12. 综合反馈
13. 城市^2
14. 城市 lon
15. 城市 lat
16. 城市 物 业 费
17. 城市 燃气费
18. 城市 停车位
19. 城市 coord_x
20. 城市 coord_y
21. 城市 房龄
22. 城市 客厅数
23. 城市 总楼层
24. 城市 户均楼栋房屋数
25. 面积^2
26. 面积 物 业 费
27. 面积 coord_y
28. 面积 卫生间数
29. 面积 总楼层
30. lon^2
31. lon 房屋总数
32. lon 物 业 费
33. lon 燃气费
34. lon 停车位
35. lon coord_x
36. lon 房龄
37. lon 朝向_南
38. lat 物 业 费
39. lat 建筑年份
40. lat 客厅数
41. lat 卫生间数
42. lat 朝向_南
43. lat 朝向_北
44. 房屋总数 停车位
45. 房屋总数 coord_y
46. 房屋总数 总楼层
47. 楼栋总数^2
48. 楼栋总数 容 积 率
49. 楼栋总数 coord_x
50. 楼栋总数 房龄
51. 楼栋总数 客厅数
52. 绿 化 率^2
53. 容 积 率^2
54. 容 积 率 物 业 费
55. 容 积 率 燃气费
56. 容 积 率 停车位
57. 容 积 率 coord_y
58. 容 积 率 总楼层
59. 物 业 费^2
60. 物 业 费 coord_x
61. 物 业 费 建筑年份
62. 物 业 费 房龄
63. 燃气费^2
64. 燃气费 coord_y
65. 停车位^2
66. 停车位 建筑年份
67. 停车位 总楼层
68. 停车位 户均楼栋房屋数
69. 停车费用^2
70. coord_x 户均楼栋房屋数
71. coord_y^2
72. coord_y 客厅数
73. coord_y 卫生间数
74. 建筑年份 卧室数
75. 卧室数^2
76. 卧室数 客厅数
77. 卧室数 卫生间数
78. 卧室数 户均楼栋房屋数
79. 卫生间数^2
80. 总楼

In [10]:
import joblib 

train_keep_cols = keep_cols_df1  


train_preprocessor = preprocessor  

# 3. 保存训练集的多项式特征生成器和特征选择器
train_poly = poly  
train_selector = selector  


train_numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()
train_medians = df1[train_numeric_cols].median()  


# 1. 保存训练集筛选列
joblib.dump(train_keep_cols, "train_keep_cols.pkl")

# 2. 保存预处理组件
joblib.dump(train_preprocessor, "train_preprocessor.pkl")

# 3. 保存多项式特征生成器
joblib.dump(train_poly, "train_poly.pkl")

# 4. 保存特征选择器
joblib.dump(train_selector, "train_selector.pkl")

# 5. 保存训练集中位数
joblib.dump(train_medians, "train_medians.pkl")

# 6. 保存训练集最终特征列名
train_processed_columns = processed_columns  
joblib.dump(train_processed_columns, "train_processed_columns.pkl")


print("所有训练集配置已保存为.pkl文件！")

所有训练集配置已保存为.pkl文件！


In [11]:

models = {
    "OLS": LinearRegression(),
    "Lasso": Lasso(alpha=0.01, random_state=111),
    "Ridge": Ridge(alpha=1.0, random_state=111),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=111),
    "RandomForest": RandomForestRegressor(random_state=111)  # 新增：随机森林模型
}

ridge_params = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(
    models["Ridge"],  
    ridge_params,
    cv=6,  
    scoring='neg_mean_absolute_error',
    n_jobs=1  
)


ridge_grid.fit(X_train_selected, y_train_clean)
best_ridge = ridge_grid.best_estimator_

# 打印最优参数
print("\nRidge Best Parameters:", ridge_grid.best_params_)
# ---------- 新增：训练并评估随机森林 ----------
# 1. 取出随机森林模型
rf_model = models["RandomForest"]

# 2. 用筛选后的特征训练模型（和之前Ridge的训练数据一致）
rf_model.fit(X_train_selected, y_train_clean)

# 3. 预测（训练集+测试集）
y_train_pred_rf = rf_model.predict(X_train_selected)
y_test_pred_rf = rf_model.predict(X_test_selected)


Ridge Best Parameters: {'alpha': 0.001}



KeyboardInterrupt



In [63]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)  
    mse = mean_squared_error(y_true, y_pred)   
    rmse = np.sqrt(mse)                            
    return mae, rmse
rf_train_mae, rf_train_rmse = evaluate_model(y_train_clean, y_train_pred_rf)
rf_test_mae, rf_test_rmse = evaluate_model(y_test, y_test_pred_rf)

# 打印结果
print("\n==== 随机森林模型评估结果 ====")
print(f"训练集MAE: {rf_train_mae:.4f}, 训练集RMSE: {rf_train_rmse:.4f}")
print(f"测试集MAE: {rf_test_mae:.4f}, 测试集RMSE: {rf_test_rmse:.4f}")

rf_results = {
    "In-sample MAE": rf_train_mae,       
    "In-sample RMAE": rf_train_rmse,     
    "Out-of-sample MAE": rf_test_mae,    
    "Out-of-sample RMAE": rf_test_rmse   

}

import joblib  
joblib.dump(rf_model, "trained_rf_model.pkl") 
print("随机森林模型已保存为：trained_rf_model.pkl")


==== 随机森林模型评估结果 ====
训练集MAE: 0.0455, 训练集RMSE: 0.0672
测试集MAE: 0.1279, 测试集RMSE: 0.1919
随机森林模型已保存为：trained_rf_model.pkl


In [12]:

def evaluate_model(model, X_train, y_train, X_test, y_test):
    """计算模型的MAE/RMAE（样本内/外+交叉验证）"""

    y_pred_train = model.predict(X_train)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmae_train = np.sqrt(mae_train)
    

    y_pred_test = model.predict(X_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmae_test = np.sqrt(mae_test)
    

    cv_scores = cross_val_score(
        model, X_train, y_train, cv=6, scoring="neg_mean_absolute_error"
    )
    mae_cv = -cv_scores.mean()
    rmae_cv = np.sqrt(mae_cv)
    
    return {
        "In-sample MAE": mae_train,
        "In-sample RMAE": rmae_train,
        "Out-of-sample MAE": mae_test,
        "Out-of-sample RMAE": rmae_test,
        "6-fold CV MAE": mae_cv,
        "6-fold CV RMAE": rmae_cv
    }



ridge_results = evaluate_model(best_ridge, X_train_selected, y_train_clean, X_test_selected, y_test)
print("\nRidge模型性能:")
for metric, val in ridge_results.items():
    print(f"{metric}: {val:.4f}")



ols = LinearRegression()
ols.fit(X_train_selected, y_train_clean)
ols_results = evaluate_model(ols, X_train_selected, y_train_clean, X_test_selected, y_test)

print("\nOLS模型性能:")
for metric, val in ols_results.items():
    print(f"{metric}: {val:.4f}")


elastic_net = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=111)
elastic_net.fit(X_train_selected, y_train_clean)


elastic_results = evaluate_model(elastic_net, X_train_selected, y_train_clean, X_test_selected, y_test)


print("\nElasticNet模型性能:")
for metric, val in elastic_results.items():
    print(f"{metric}: {val:.4f}")

all_results = {
    "OLS": ols_results,
    "Ridge（最优）": ridge_results,
    "ElasticNet": elastic_results, 
    "随机森林": rf_results 
}

results_df = pd.DataFrame(all_results).T  
print("\n所有模型性能对比:")
print(results_df.round(4))  



print(f"\n异常值处理后训练集样本数：{X_train_clean.shape[0]}")


Ridge模型性能:
In-sample MAE: 0.3206
In-sample RMAE: 0.5662
Out-of-sample MAE: 0.3245
Out-of-sample RMAE: 0.5696
6-fold CV MAE: 0.3211
6-fold CV RMAE: 0.5666

OLS模型性能:
In-sample MAE: 0.3206
In-sample RMAE: 0.5662
Out-of-sample MAE: 0.3245
Out-of-sample RMAE: 0.5696
6-fold CV MAE: 0.3211
6-fold CV RMAE: 0.5666

ElasticNet模型性能:
In-sample MAE: 0.3349
In-sample RMAE: 0.5787
Out-of-sample MAE: 0.3406
Out-of-sample RMAE: 0.5836
6-fold CV MAE: 0.3353
6-fold CV RMAE: 0.5791


NameError: name 'rf_results' is not defined

In [18]:

import pandas as pd
import numpy as np
import re
import joblib
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer


train_keep_cols = joblib.load("train_keep_cols.pkl")  
train_preprocessor = joblib.load("train_preprocessor.pkl") 
train_poly = joblib.load("train_poly.pkl")
train_selector = joblib.load("train_selector.pkl")
train_medians = joblib.load("train_medians.pkl")
train_processed_columns = joblib.load("train_processed_columns.pkl")

rf_model = joblib.load("trained_rf_model.pkl")

In [19]:
# 1. 数据处理
data_rent =r"D:\人工智能\Python exam\ruc_Class25Q2_test_rent.csv"
df1 = pd.read_csv(data_rent, dtype=str,encoding="gbk")
#print("===== df1 原始数据基本信息 =====")
#print(df1.info()) 
non_null_counts_df1 = df1.notnull().sum()
keep_cols_df1 = non_null_counts_df1[non_null_counts_df1 > 6000].index.tolist()
df1 = df1[keep_cols_df1]
#print("\n===== df1 筛选后数据基本信息 =====")
#print(df1.info())
#df1 = df1.dropna()  
#df1 = df1.drop_duplicates()  
#print(df1.info())

df1['交易时间'] = pd.to_datetime(df1['交易时间'], errors='coerce')
df1['交易年份'] = df1['交易时间'].dt.year

def parse_build_year(year_str):

    year_str = str(year_str)

    years = re.findall(r'\d{4}', year_str)
    
    if years:

        return np.mean([int(y) for y in years])

    return np.nan 


df1['建筑年份'] = df1['建筑年代'].apply(parse_build_year)

df1['房龄'] = df1['交易年份'] - df1['建筑年份']

df1['卧室数'] = df1['户型'].astype(str).str.extract(r'(\d+)[室房]').fillna(0).astype(int)
df1['客厅数'] = df1['户型'].astype(str).str.extract(r'(\d+)厅').fillna(0).astype(int)
df1['卫生间数'] = df1['户型'].astype(str).str.extract(r'(\d+)卫').fillna(0).astype(int)


s = df1['楼层'].astype(str)
df1['总楼层'] = df1['楼层'].astype(str).str.extract(r'(\d+)层').squeeze()

df1['总楼层'] = pd.to_numeric(df1['总楼层'], errors='coerce')
df1['初始楼层类型'] = s.str.extract(r'^(.*?)/').squeeze()

df1['楼层类型'] = s.str.extract(r'^(低楼层|中楼层|高楼层)').squeeze()
df1['楼层类型'] = df1['楼层类型'].fillna('普通楼层')

df1['楼层类型'] = df1.apply(classify_common_floor, axis=1)
numeric_cols = [ '面积', 'lon', 'lat', 'coord_x', 'coord_y','城市','房屋总数','楼栋总数','绿 化 率','容 积 率','物 业 费','燃气费','停车位','停车费用']

def process_range_text(text):

    text_str = str(text)
    cleaned_text = re.sub(r"[^\d.-]", "", text_str)
    

    if "-" in cleaned_text:

        num1, num2 = cleaned_text.split("-")
        return (float(num1) + float(num2)) / 2

    else:

        return float(cleaned_text) if cleaned_text.strip() else np.nan

for col in numeric_cols:
    
    if col == '面积':

        df1[col] = df1[col].str.replace('㎡', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '房屋总数':

        df1[col] = df1[col].str.replace('户', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '楼栋总数':
 
        df1[col] = df1[col].str.replace('栋', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '绿化率':

        df1[col] = df1[col].str.replace('%', '')
        df1[col] = df1[col].apply(process_range_text)
        df1[col] = df1[col] / 100  
    
    else:

        df1[col] = df1[col].apply(process_range_text)
df1['户均楼栋房屋数'] = df1['房屋总数'] / df1['楼栋总数']
df1['每户停车位'] = df1['停车位'] / df1['房屋总数']

df1['朝向'] = df1['朝向'].astype(str).str.replace(' ', '')  


base_directions = ['东', '南', '西', '北']
for direction in base_directions:

    df1[f'朝向_{direction}'] = df1['朝向'].str.contains(direction, na=False).astype(int)


df1['朝向'] = (df1['朝向'] == '未知').astype(int)

if all(col in df1.columns for col in ['核心卖点', '户型介绍', '周边配套']):
        df1['description_combined'] = (
            df1['核心卖点'].fillna('') + 
            df1['户型介绍'].fillna('') + 
            df1['周边配套'].fillna('')
        )
        
        
        objective_keywords = ['户型方正', '人车分流', '学区', '地铁', '医院', '商场', '超市', '公园', '菜市场']
        for keyword in objective_keywords:
            df1[f'Desc_{keyword}'] = df1['description_combined'].str.contains(keyword, na=False).astype(int)

if '客户反馈' in df1.columns:
        positive_keywords = ['体验佳', '干净', '安静', '方便', '采光好', '物业好', '安全','好', '安全', '阳光充足', '整洁']
        negative_keywords = ['老旧', '费高', '噪音', '通风差', '潮湿', '漏水', '老化', '乱', '卫生差','一般']
        
        df1['积极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in positive_keywords if word in x)
        )
        df1['消极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in negative_keywords if word in x)
        )
        df1['综合反馈'] = df1['积极反馈'] - df1['消极反馈']

numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()
df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())
print("===== 数值列转换后信息 =====")
#print(df1.info())
#print(df1.head(15)) 

#print("\n数据统计描述：")
#print(df1.describe())  
#print("\n前5行数据：")
print(df1.head(8)) 
df_test = df1  
df_test['面积'] = np.log(df_test['面积'].astype(float) + 1e-6)

===== 数值列转换后信息 =====
        ID    城市      户型   装修       楼层      面积  朝向       交易时间 付款方式 租赁方式  ...  \
0  2000000   1.0  2室2厅1卫  精装修  低楼层/18层   86.94   0 2025-08-01  NaN   整租  ...   
1  2000001  10.0  2室1厅1卫  精装修   低楼层/8层   72.60   0 2025-05-23  月付价   整租  ...   
2  2000002   3.0  2室2厅1卫  精装修  中楼层/20层   98.00   0 2025-02-18  NaN   整租  ...   
3  2000003   0.0  2室1厅1卫  精装修  高楼层/12层   98.97   0 2025-02-17  季付价   整租  ...   
4  2000004   3.0  3室2厅2卫  精装修  中楼层/23层  170.53   0 2025-03-24  季付价   整租  ...   
5  2000005   2.0  3室2厅1卫  精装修   30/45层   86.77   0 2025-05-13  季付价   整租  ...   
6  2000006   9.0  3室2厅2卫  NaN  中楼层/34层  135.00   0 2025-04-14  NaN   整租  ...   
7  2000007   4.0  3室1厅1卫  精装修   中楼层/6层   52.09   0 2025-04-29  季付价   整租  ...   

  楼层类型     户均楼栋房屋数     每户停车位 朝向_东 朝向_南  朝向_西  朝向_北 积极反馈 消极反馈 综合反馈  
0  低楼层   82.666667  1.612903    0    1     0     1    0    0    0  
1  低楼层   53.733333  0.248139    0    1     0     0    2    0    2  
2  中楼层  102.076923  0.543075    0    1     0     0    

In [20]:

print("训练集处理后的列 (train_processed_columns):")
print(f"数量: {len(train_processed_columns)}")
print(f"具体列: {train_processed_columns}")


print("\n测试集业务筛选后的列 (df_test.columns):")
print(f"数量: {len(df_test.columns)}")
print(f"具体列: {list(df_test.columns)}")


missing_cols = set(train_processed_columns) - set(df_test.columns)
print(f"\n缺失的列: {missing_cols}")
print(f"缺失数量: {len(missing_cols)}")

df_test = df_test[train_processed_columns]  
print(f"测试集筛选列后形状：{df_test.shape}")

print("测试集特征变量的列名：")
print(df_test.columns.tolist())  


valid_processed_columns = [
    col for col in train_processed_columns  
    if col in df_test.columns and col != "Price"  
]


df_test_features = df_test[valid_processed_columns].copy()
print(f"测试集对齐后特征列：{df_test_features.columns.tolist()}")
print(f"测试集对齐后形状：{df_test_features.shape}")  

训练集处理后的列 (train_processed_columns):
数量: 30
具体列: ['城市', '面积', 'lon', 'lat', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '燃气费', '停车位', '停车费用', 'coord_x', 'coord_y', '建筑年份', '房龄', '卧室数', '客厅数', '卫生间数', '总楼层', '户均楼栋房屋数', '每户停车位', '积极反馈', '消极反馈', '综合反馈', '朝向', '朝向_东', '朝向_南', '朝向_西', '朝向_北']

测试集业务筛选后的列 (df_test.columns):
数量: 57
具体列: ['ID', '城市', '户型', '装修', '楼层', '面积', '朝向', '交易时间', '付款方式', '租赁方式', '电梯', '用水', '用电', '燃气', '配套设施', 'lon', 'lat', '年份', '区县', '板块', '物业类别', '建筑年代', '开发商', '房屋总数', '楼栋总数', '物业公司', '绿 化 率', '容 积 率', '物 业 费', '建筑结构', '产权描述', '供水', '供电', '燃气费', '停车位', '停车费用', 'coord_x', 'coord_y', '客户反馈', '交易年份', '建筑年份', '房龄', '卧室数', '客厅数', '卫生间数', '总楼层', '初始楼层类型', '楼层类型', '户均楼栋房屋数', '每户停车位', '朝向_东', '朝向_南', '朝向_西', '朝向_北', '积极反馈', '消极反馈', '综合反馈']

缺失的列: set()
缺失数量: 0
测试集筛选列后形状：(9773, 30)
测试集特征变量的列名：
['城市', '面积', 'lon', 'lat', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '燃气费', '停车位', '停车费用', 'coord_x', 'coord_y', '建筑年份', '房龄', '卧室数', '客厅数', '卫生间数', '总楼层', '户均楼栋房屋数', '每户停车位', '积极反馈', '

In [21]:


valid_processed_columns = [
    col for col in train_processed_columns  
    if col in df_test.columns and col != "Price"  
]


df_test_features = df_test[valid_processed_columns].copy()
print(f"测试集对齐后特征列：{df_test_features.columns.tolist()}")
print(f"测试集对齐后形状：{df_test_features.shape}")  


测试集对齐后特征列：['城市', '面积', 'lon', 'lat', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '燃气费', '停车位', '停车费用', 'coord_x', 'coord_y', '建筑年份', '房龄', '卧室数', '客厅数', '卫生间数', '总楼层', '户均楼栋房屋数', '每户停车位', '积极反馈', '消极反馈', '综合反馈', '朝向', '朝向_东', '朝向_南', '朝向_西', '朝向_北']
测试集对齐后形状：(9773, 30)


In [22]:

X_test_processed = train_preprocessor.transform(df_test_features)
print(f"测试集预处理后形状：{X_test_processed.shape}")  

测试集预处理后形状：(9773, 30)


In [23]:

X_test_poly = train_poly.transform(X_test_processed)
print(f"测试集多项式特征后形状：{X_test_poly.shape}")  

X_test_selected = train_selector.transform(X_test_poly)
print(f"测试集特征选择后形状：{X_test_selected.shape}")  

测试集多项式特征后形状：(9773, 495)
测试集特征选择后形状：(9773, 81)


In [24]:

#y_test_pred_log = rf_model.predict(X_test_selected)
y_test_pred_log = ols.predict(X_test_selected)
print(f"预测的log_Price前5个值：{y_test_pred_log[:5]}")

预测的log_Price前5个值：[12.23341736 13.02928943 13.25134867 13.76296619 13.82889806]


In [25]:

#y_test_pred = np.exp(y_test_pred_log) * 1000000 
y_test_pred = np.exp(y_test_pred_log) 


submission_df = pd.DataFrame({
    "ID": df1["ID"],  
    "预测租金（元）": y_test_pred.astype(int)  
})


submission_df.to_csv("test_rent_price_prediction.csv", index=False, encoding="utf-8-sig")
print("预测结果已保存为：test_rent_price_prediction.csv")


print("\n测试集前10条预测结果：")
print(submission_df.head(10))

预测结果已保存为：test_rent_price_prediction.csv

测试集前10条预测结果：
        ID  预测租金（元）
0  2000000   205544
1  2000001   455563
2  2000002   568836
3  2000003   948812
4  2000004  1013477
5  2000005   250117
6  2000006   250670
7  2000007        0
8  2000008   203249
9  2000009   333002
